In [0]:
CREATE LIVE TABLE dq_metrics
COMMENT "Central Data Quality metrics for dashboarding calculated from final physical tables"
AS 
WITH 
cust_stats AS (
  SELECT 'customers' AS table_name,
  (SELECT COUNT(*) FROM live.silver_customers) + (SELECT COUNT(*) FROM live.quarantine_customers) AS records_checked,
  (SELECT COUNT(*) FROM live.quarantine_customers) AS hard_failed_records,
  (SELECT COUNT(*) FROM live.silver_customers WHERE dq_flags != '') AS warn_failed_records
),
store_stats AS (
  SELECT 'stores' AS table_name,
  (SELECT COUNT(*) FROM live.silver_stores) + (SELECT COUNT(*) FROM live.quarantine_stores) AS records_checked,
  (SELECT COUNT(*) FROM live.quarantine_stores) AS hard_failed_records,
  (SELECT COUNT(*) FROM live.silver_stores WHERE dq_flags != '') AS warn_failed_records
),
inv_stats AS (
  SELECT 'inventory' AS table_name,
  (SELECT COUNT(*) FROM live.silver_inventory) + (SELECT COUNT(*) FROM live.quarantine_inventory) AS records_checked,
  (SELECT COUNT(*) FROM live.quarantine_inventory) AS hard_failed_records,
  (SELECT COUNT(*) FROM live.silver_inventory WHERE dq_flags != '') AS warn_failed_records
),
prod_stats AS (
  SELECT 'products' AS table_name,
  (SELECT COUNT(*) FROM live.silver_products) + (SELECT COUNT(*) FROM live.quarantine_products) AS records_checked,
  (SELECT COUNT(*) FROM live.quarantine_products) AS hard_failed_records,
  (SELECT COUNT(*) FROM live.silver_products WHERE dq_flags != '') AS warn_failed_records
),
promo_stats AS (
  SELECT 'promotions' AS table_name,
  (SELECT COUNT(*) FROM live.silver_promotions) + (SELECT COUNT(*) FROM live.quarantine_promotions) AS records_checked,
  (SELECT COUNT(*) FROM live.quarantine_promotions) AS hard_failed_records,
  (SELECT COUNT(*) FROM live.silver_promotions WHERE dq_flags != '') AS warn_failed_records
),
region_stats AS (
  SELECT 'regions' AS table_name,
  (SELECT COUNT(*) FROM live.silver_regions) + (SELECT COUNT(*) FROM live.quarantine_regions) AS records_checked,
  (SELECT COUNT(*) FROM live.quarantine_regions) AS hard_failed_records,
  (SELECT COUNT(*) FROM live.silver_regions WHERE dq_flags != '') AS warn_failed_records
),
return_stats AS (
  SELECT 'returns' AS table_name,
  (SELECT COUNT(*) FROM live.silver_returns) + (SELECT COUNT(*) FROM live.quarantine_returns) AS records_checked,
  (SELECT COUNT(*) FROM live.quarantine_returns) AS hard_failed_records,
  (SELECT COUNT(*) FROM live.silver_returns WHERE dq_flags != '') AS warn_failed_records
),
txn_stats AS (
  SELECT 'transactions' AS table_name,
  (SELECT COUNT(*) FROM live.silver_transactions) + (SELECT COUNT(*) FROM live.quarantine_transactions) AS records_checked,
  (SELECT COUNT(*) FROM live.quarantine_transactions) AS hard_failed_records,
  (SELECT COUNT(*) FROM live.silver_transactions WHERE dq_flags != '') AS warn_failed_records
),
cal_stats AS (
  -- Calendar only has hard failures (no warn flags)
  SELECT 'calendar' AS table_name,
  (SELECT COUNT(*) FROM live.silver_calendar) + (SELECT COUNT(*) FROM live.quarantine_calendar) AS records_checked,
  (SELECT COUNT(*) FROM live.quarantine_calendar) AS hard_failed_records,
  0 AS warn_failed_records
),
all_stats AS (
  SELECT * FROM cust_stats UNION ALL
  SELECT * FROM store_stats UNION ALL
  SELECT * FROM inv_stats UNION ALL
  SELECT * FROM prod_stats UNION ALL
  SELECT * FROM promo_stats UNION ALL
  SELECT * FROM region_stats UNION ALL
  SELECT * FROM return_stats UNION ALL
  SELECT * FROM txn_stats UNION ALL
  SELECT * FROM cal_stats
)
SELECT 
  table_name,
  records_checked,
  hard_failed_records,
  warn_failed_records,
  ROUND(hard_failed_records / NULLIF(records_checked, 0) * 100, 2) AS failure_percentage,
  current_timestamp() AS run_time
FROM all_stats;